# Lab 5.1 &mdash; A Support Desk That Triages Itself

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Build the supervisor/worker graph: one queue in, three specialists, one reply out
- Wire the supervisor as a real <code>add_conditional_edges</code>, not an <code>if</code> statement
- Score the router against twelve labelled tickets &mdash; a supervisor is a classifier
- Swap in a model-routed supervisor and score that on the same twelve

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a routing key), so they are deterministic
> and never depend on the model. Cells marked **Run it for real** put your work in front of
> the sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **The system on slide 2.** Tickets arrive on one queue; a supervisor reads each one and
> picks a specialist; one specialist runs; a `resolve` node writes the reply.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# A customer support desk for a SaaS product. One queue in, three specialists behind it.
# This case file is Module 5 lab 5.1 only -- 5.2 and 5.3 are different systems.

SPECIALISTS = ("billing", "tech", "account")

DESK = {
    "billing": "Charges, refunds, invoices, plan and price changes.",
    "tech":    "Errors, outages, failing API calls, anything broken.",
    "account": "Seats, owners, permissions, sign-in and access.",
}

# Twelve tickets with a known correct specialist. The last four name no keyword at all --
# their intent is only implied, which is exactly where a rule table runs out and the reason
# anyone reaches for a model.
TICKETS = [
    ("I was charged twice for March.",                              "billing"),
    ("Can I get an invoice with our VAT number on it?",             "billing"),
    ("We want to downgrade to the starter plan.",                   "billing"),
    ("Your API returns 500 on every /sync call since 09:00.",       "tech"),
    ("The export button throws an error and nothing downloads.",    "tech"),
    ("Webhooks stopped firing after your deploy.",                  "tech"),
    ("Please add two more seats for the new joiners.",              "account"),
    ("Move the workspace owner to priya@example.com.",              "account"),
    # intent implied, no keyword names it
    ("Nobody on my team can get in this morning.",                  "account"),
    ("We were told this would be free until June.",                 "billing"),
    ("Everything was fine yesterday and now nothing loads.",        "tech"),
    ("Someone who left in May can still see our data.",             "account"),
]

print(f"{len(TICKETS)} tickets, {len(DESK)} specialists")

## Concept

A supervisor decides which specialist handles a request. In LangGraph that is one thing: a
**conditional edge** out of a supervisor node.

`add_conditional_edges(source, fn, path_map)` needs two different things, and people mix them up:

| | |
|---|---|
| `fn` | a function of **state** that returns a **key** |
| `path_map` | `{key: node name}` &mdash; which node each key means |

Which makes the supervisor a classifier with a known correct answer. So it has an accuracy, and
almost nobody measures it &mdash; even though a misroute wastes every token spent downstream of it.

## Section 1 &mdash; The graph

Three worker nodes, a supervisor node that writes its decision **into state**, and a `resolve`
node that all three feed. Two decisions are yours: the **fallback** (every ticket the keyword
table does not recognise ends up there, which makes that one line the router's whole failure
mode) and which function is the **routing adapter**.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END


class DeskState(TypedDict):
    ticket: str                       # what the customer wrote
    route: str | None                 # the supervisor's decision, readable afterwards
    answer: str | None                # the specialist's reply
    trail: Annotated[list, add]       # append: every node leaves a mark


# Order matters: the most specific group goes first.
KEYWORDS = [
    ("billing", ("charge", "charged", "invoice", "refund", "plan", "price", "vat")),
    ("tech",    ("error", "500", "api", "webhook", "broken", "throws", "down")),
    ("account", ("seat", "owner", "permission", "access", "sign-in")),
]

def route_by_keyword(ticket: str) -> str:
    """The first keyword group that matches wins."""
    low = (ticket or "").lower()
    for specialist, words in KEYWORDS:
        if any(w in low for w in words):
            return specialist
    # Nothing matched. Tech is the least damaging place to land a stranger: it reads the
    # ticket and can hand it on, where billing would be answering about money it has not read.
    return "tech"


def supervisor(state: DeskState) -> dict:
    """A node like any other. It decides, and it writes the decision down."""
    choice = route_by_keyword(state["ticket"])
    return {"route": choice, "trail": [f"supervisor -> {choice}"]}


def make_worker(name: str):
    """Three specialists that differ only in who they are. Deterministic, so the graph
    can be asserted exactly offline; the model shows up in the live cell below."""
    def worker(state: DeskState) -> dict:
        return {"answer": f"[{name}] {DESK[name]} Re: {state['ticket'][:40]}",
                "trail": [f"{name} handled it"]}
    return worker


def resolve(state: DeskState) -> dict:
    return {"trail": ["resolved"]}


def pick_specialist(state: DeskState) -> str:
    """The adapter: takes STATE, returns a KEY of the path map."""
    return state["route"]


def build_desk():
    g = StateGraph(DeskState)
    g.add_node("supervisor", supervisor)
    for name in SPECIALISTS:
        g.add_node(name, make_worker(name))
    g.add_node("resolve", resolve)

    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", pick_specialist, {n: n for n in SPECIALISTS})
    for name in SPECIALISTS:
        g.add_edge(name, "resolve")
    g.add_edge("resolve", END)
    return g.compile()


def fresh(ticket: str) -> dict:
    return {"ticket": ticket, "route": None, "answer": None, "trail": []}

In [ ]:
# --- Self-check: Section 1   (a REAL compiled graph, really running -- still no model)
def _run(ticket: str) -> dict:
    return build_desk().invoke(fresh(ticket))

check("the desk compiles",
      lambda: build_desk() is not None)
check("the fallback is a specialist that actually exists",
      lambda: route_by_keyword("zzzz nothing here zzzz") in SPECIALISTS,
      "a conditional edge returning a key the path map does not have is a runtime error")
check("a billing ticket reaches the billing agent",
      lambda: _run("I was charged twice for March.")["route"] == "billing")
check("an outage reaches the tech agent",
      lambda: _run("Your API returns 500 on every /sync call.")["route"] == "tech")
check("the decision is written into state, not hidden in control flow",
      lambda: _run("Please add two more seats.")["route"] == "account",
      "a routing decision you cannot read back afterwards is one you cannot audit")
check("exactly ONE specialist runs per ticket",
      lambda: sum(1 for m in _run("Please add two more seats.")["trail"]
                  if "handled it" in m) == 1,
      "a conditional edge picks one path; fanning out to all three is Lab 5.2's shape")
check("and every ticket still reaches resolve",
      lambda: _run("Everything was fine yesterday.")["trail"][-1] == "resolved")

def _trace():
    for chunk in build_desk().stream(fresh("Move the workspace owner to priya@example.com.")):
        for node, update in chunk.items():
            print(f"  {node:12} -> {list(update)}")
guard(_trace)

## Section 2 &mdash; Score the supervisor

Twelve tickets with a known correct specialist. The harness is given &mdash; nothing in it is a
design decision. What *is* a decision is the last line: the bar a router has to clear before you
would put it in front of customers. Pick a number you would defend in a review, not one that
makes your router pass.

In [ ]:
def selections(router) -> dict:
    """{ticket: chosen specialist} over the whole eval set."""
    return {ticket: router(ticket) for ticket, _ in TICKETS}


def accuracy(sel: dict) -> float:
    """Fraction routed to the expected specialist. No selection counts as wrong."""
    return sum(1 for t, want in TICKETS if sel.get(t) == want) / len(TICKETS)


def confusion(sel: dict) -> dict:
    """{(expected, chosen): count} over the misses only."""
    out = {}
    for ticket, want in TICKETS:
        got = sel.get(ticket)
        if got != want:
            out[(want, got)] = out.get((want, got), 0) + 1
    return out


def clears_the_bar(acc: float) -> bool:
    """Would you ship a supervisor that routes this well? Decide the bar and defend it.

    0.85 here: at 12 tickets that is one miss allowed, and a support desk can absorb one
    handoff in eight. Below that the specialists spend their day forwarding.
    """
    return acc >= 0.85


def _report():
    sel = selections(route_by_keyword)
    print(f"rule-based supervisor: {accuracy(sel):.0%} on {len(TICKETS)} tickets\n")
    for (want, got), n in sorted(confusion(sel).items(), key=lambda kv: -kv[1]):
        print(f"  {n}x  should have been {want:8} -> went to {got}")
guard(_report)

In [ ]:
# --- Self-check: Section 2
_rule = None
def rule_selections():
    global _rule
    if _rule is None:
        _rule = selections(route_by_keyword)
    return _rule

check("the eval set covers every specialist",
      lambda: {w for _, w in TICKETS} == set(SPECIALISTS))
check("it contains tickets whose intent is only implied",
      lambda: sum(1 for t, _ in TICKETS
                  if not any(w in t.lower() for _, ws in KEYWORDS for w in ws)) >= 3,
      "an eval set of keyword-shaped tickets measures the keywords, not the routing")
check("the rule supervisor gets most of it right",
      lambda: accuracy(rule_selections()) > 0.6)
check("but not all of it -- there is headroom to argue about",
      lambda: accuracy(rule_selections()) < 1.0)
check("every miss lands on the FALLBACK, not on a random specialist",
      lambda: {got for _, got in confusion(rule_selections())} == {route_by_keyword("zzzz")},
      "a rule router's failure mode IS its fallback -- unrecognised intent all piles up there")
check("your acceptance bar is a real bar",
      lambda: clears_the_bar(1.0) and not clears_the_bar(0.5),
      "a bar nothing can clear is not a bar, and neither is one a coin flip clears")
check("and it is a number a support desk could live with",
      lambda: clears_the_bar(0.95) and not clears_the_bar(0.6),
      "at 0.6 two tickets in five reach the wrong desk and get forwarded by hand")

def _verdict():
    acc = accuracy(rule_selections())
    print(f"rule-based router: {acc:.0%} -> "
          f"{'clears your bar' if clears_the_bar(acc) else 'does NOT clear your bar'}")
guard(_verdict)

## Run it for real &mdash; a model-routed supervisor

Same graph, same eval set, same metric. The only thing that changes is the function inside
`route_with_model`. What the model gets to read is `DESK` &mdash; three one-line descriptions,
which is the entire difference between the two routers.

In [ ]:
ROUTE_SYSTEM = ("You route one customer support ticket to exactly one specialist desk. "
                "Reply with the desk's name alone -- no punctuation, no explanation.")

def route_with_model(ticket: str) -> str:
    """Ask the model to pick a desk. Anything unrecognised falls back to the keywords."""
    listing = "\n".join(f"- {n}: {d}" for n, d in DESK.items())
    reply = ask(f"Desks:\n{listing}\n\nTicket: {ticket}\n\nDesk:", system=ROUTE_SYSTEM)
    word = (reply or "").strip().strip("`.\"' ").lower().split()
    return word[0] if word and word[0] in DESK else route_by_keyword(ticket)


def _bake_off():
    rule = rule_selections()
    model = selections(route_with_model)
    print(f"{'supervisor':16}{'accuracy':>10}   clears your bar?")
    print("-" * 46)
    for label, sel in (("rule-based", rule), ("model", model)):
        acc = accuracy(sel)
        print(f"{label:16}{acc:>9.0%}   {'yes' if clears_the_bar(acc) else 'no'}")
    print()
    for (want, got), n in sorted(confusion(model).items(), key=lambda kv: -kv[1]):
        print(f"  model missed {n}x: {want} -> {got}")
    print("\nAnd the model-routed supervisor inside the real graph:")
    # The graph is unchanged. Only the function the supervisor node calls is different.
    global route_by_keyword
    keep, route_by_keyword = route_by_keyword, route_with_model
    try:
        out = build_desk().invoke(fresh("Nobody on my team can get in this morning."))
        print(f"  route={out['route']}  trail={out['trail']}")
    finally:
        route_by_keyword = keep

if llm_ready():
    guard(_bake_off)

In [ ]:
score()

## Your turn

1. The four implied-intent tickets are where the rule table loses. Add keywords until it wins
   all twelve &mdash; then write down how many keywords you added, and ask whether a desk with
   real tickets could keep that table current.
2. Route the ambiguous tickets to a model and the obvious ones to the keyword table. Score the
   hybrid. It is usually the cheapest thing that clears the bar, and nobody builds it.
3. Add a fourth key to the path map &mdash; `escalate` &mdash; for tickets the supervisor is not
   confident about, and decide what "not confident" means when the router is a keyword table.